# RED-diff / DPS Native-Count CT Single Run In Colab

This notebook keeps the **RED-diff repo algorithm framework** (`algos/dps.py`, `algos/reddiff.py`, `main.py`) and adds a native transmission-count CT degradation to it.

The CT measurement model is the same class of model as the PDHG formulation:

`counts ~ Poisson(I0 * exp(-A mu(x)))`

Important scope:

- This is a same-framework REDDIFF/DPS CT integration notebook.
- The default data path uses the original L067 `.IMA` CT slices and prepares float TIFFs.
- The default model path uses the DM4CT/LodoChallenge pixel-space CT diffusers checkpoint.
- DPS guidance is mapped through `algo.grad_term_weight`; `algo.eta` is kept as DDIM stochasticity.
- For CT, saved `_deg.png` files visualize the normalized log-sinogram, not the raw count tensor.


In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/daps_test.git"  #@param {type:"string"}
REPO_BRANCH = "codex-reddiff-colab-operators"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/RED-diff.zip"  #@param {type:"string"}

REPO_DIR = "/content/RED-diff"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/reddiff_ct_native_single_run_exports"  #@param {type:"string"}
DRIVE_CT_DATA_DIR = ""  #@param {type:"string"}
DM4CT_MODEL_REF = "jiayangshi/lodochallenge_pixel_diffusion"  #@param {type:"string"}

RUN_NAME = "REDDiff_DPS_CT_Native_Counts"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
BASE_CONFIG_NAME = "dm4ct_ct512_uncond"  #@param ["dm4ct_ct512_uncond"]
ALGO_NAME = "dps"  #@param ["dps", "reddiff"]

SEED = 99  #@param {type:"integer"}
TOTAL_IMAGES = 3  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}
NUM_WORKERS = 0  #@param {type:"integer"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}
SAMPLING_METRIC_EVERY = 1  #@param {type:"integer"}
PROGRESS_JSON_EVERY = 1  #@param {type:"integer"}

PREP_OUTPUT_DIR = "/content/reddiff_ct_native_preprocessed"  #@param {type:"string"}
CT_VALUE_MIN = ""  #@param {type:"string"}
CT_VALUE_MAX = ""  #@param {type:"string"}
VALID_EXTENSIONS = ".tif,.tiff,.png,.jpg,.jpeg,.dcm,.ima"  #@param {type:"string"}

# DDPM/DDIM schedule used by this RED-diff repo.
NUM_STEPS = 1000  #@param {type:"integer"}

# Native-count CT acquisition.
I0 = 10000.0  #@param {type:"number"}
NUM_ANGLES = 80  #@param {type:"integer"}
NUM_DETECTORS = 512  #@param {type:"integer"}
ATTENUATION_MIN = 0.0  #@param {type:"number"}
ATTENUATION_MAX = 1.0  #@param {type:"number"}
CT_LOSS_REDUCTION = "mean"  #@param ["mean", "sum"]

# DPS settings in this RED-diff repo.
DPS_GRAD_TERM_WEIGHT = 1.0  #@param {type:"number"}
DPS_DDIM_ETA = 0.0  #@param {type:"number"}

# REDDIFF settings in this RED-diff repo.
REDDIFF_GRAD_TERM_WEIGHT = 0.25  #@param {type:"number"}
REDDIFF_OBS_WEIGHT = 1.0  #@param {type:"number"}
REDDIFF_LR = 0.01  #@param {type:"number"}
REDDIFF_DDIM_ETA = 0.0  #@param {type:"number"}
REDDIFF_DENOISE_TERM_WEIGHT = "linear"  #@param ["linear", "sqrt", "log", "square", "trunc_linear", "const", "power2over3"]
REDDIFF_SIGMA_X0 = 0.0001  #@param {type:"number"}

SAVE_ORI = True  #@param {type:"boolean"}
SAVE_DEG = True  #@param {type:"boolean"}
SAVE_EVOLUTION = False  #@param {type:"boolean"}
SMOKE_TEST = 0  #@param {type:"integer"}

# Optional extra Hydra overrides, separated by semicolons.
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run([
        "git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo_dir.as_posix()
    ], check=True)
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repo ready: {repo_dir}")


In [ ]:
#@title Install Colab Dependencies And Checkpoint
import os
import shutil
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
req_path = Path("requirements-colab.txt")
if req_path.exists():
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", req_path.as_posix()], check=True)
    print(f"Installed {req_path}")
else:
    fallback = [
        "hydra-core==1.3.2",
        "omegaconf==2.3.0",
        "gdown==5.2.0",
        "lmdb>=1.4,<2",
        "tensorboard>=2.14",
        "scipy==1.14.1",
        "torchmetrics>=1.4,<2",
        "pydicom>=2.4,<4",
        "diffusers>=0.27,<1",
        "accelerate>=0.29,<1",
    ]
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", *fallback], check=True)
    print("Installed fallback Colab packages")

try:
    import pydicom  # noqa: F401
    print("pydicom ready")
except ModuleNotFoundError:
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "pydicom>=2.4,<4"], check=True)
    import pydicom  # noqa: F401
    print("Installed pydicom")

try:
    import diffusers  # noqa: F401
    print("diffusers ready")
except ModuleNotFoundError:
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "diffusers>=0.27,<1", "accelerate>=0.29,<1"], check=True)
    import diffusers  # noqa: F401
    print("Installed diffusers")

print(f"DM4CT model reference: {DM4CT_MODEL_REF}")


In [ ]:
CT_DEGRADATION_CODE = '\n\nclass TransmissionCTNativeCountsH(H_functions):\n    """Native transmission-count CT degradation for REDDIFF/DPS.\n\n    Input images are expected in the RED-diff repo convention x in [-1, 1].\n    The grayscale channel is mapped to nonnegative attenuation, projected with a\n    differentiable rotate/sum projector, and measured as Poisson transmission\n    counts: y ~ Poisson(I0 * exp(-A mu(x))).\n    """\n\n    measurement_model = "transmission_ct_native_counts"\n\n    def __init__(\n        self,\n        channels,\n        img_dim,\n        num_angles=80,\n        num_detectors=None,\n        I0=10000.0,\n        attenuation_min=0.0,\n        attenuation_max=1.0,\n        detector_scale=1.0,\n        angle_offset_deg=0.0,\n        loss_reduction="mean",\n        device="cuda",\n    ):\n        self.channels = int(channels)\n        self.img_dim = int(img_dim)\n        self.num_angles = int(num_angles)\n        self.num_detectors = int(num_detectors) if num_detectors is not None else int(img_dim)\n        self.I0 = float(I0)\n        self.attenuation_min = float(attenuation_min)\n        self.attenuation_max = float(attenuation_max)\n        self.detector_scale = float(detector_scale)\n        self.angle_offset_deg = float(angle_offset_deg)\n        self.loss_reduction = str(loss_reduction)\n        self.device = torch.device(device)\n        self.sigma = 0.0\n\n        angles = torch.arange(self.num_angles, dtype=torch.float32, device=self.device)\n        angles = angles * (np.pi / max(1, self.num_angles))\n        angles = angles + np.deg2rad(self.angle_offset_deg)\n        self.angles = angles\n\n    def _as_gray(self, image):\n        if image.dim() != 4:\n            raise ValueError(f"Expected [B,C,H,W], got {tuple(image.shape)}")\n        if image.shape[1] == 1:\n            return image\n        return image.mean(dim=1, keepdim=True)\n\n    def _repeat_channels(self, image):\n        if self.channels == 1:\n            return image\n        return image.repeat(1, self.channels, 1, 1)\n\n    def _mu(self, image):\n        gray = self._as_gray(image)\n        x01 = ((gray + 1.0) * 0.5).clamp(0.0, 1.0)\n        return self.attenuation_min + x01 * (self.attenuation_max - self.attenuation_min)\n\n    def _resize_detector_axis(self, sino, num_detectors):\n        if sino.shape[-1] == num_detectors:\n            return sino\n        b, c, a, d = sino.shape\n        flat = sino.reshape(b * c * a, 1, d)\n        flat = F.interpolate(flat, size=num_detectors, mode="linear", align_corners=False)\n        return flat.reshape(b, c, a, num_detectors)\n\n    def _forward_mu(self, mu):\n        b, c, h, w = mu.shape\n        projections = []\n        dx = 2.0 / max(h, 1)\n        angles = self.angles.to(device=mu.device, dtype=mu.dtype)\n        for angle in angles:\n            cos_a = torch.cos(angle)\n            sin_a = torch.sin(angle)\n            theta = torch.stack(\n                [\n                    torch.stack([cos_a, -sin_a, torch.zeros_like(cos_a)]),\n                    torch.stack([sin_a, cos_a, torch.zeros_like(cos_a)]),\n                ]\n            ).unsqueeze(0).expand(b, -1, -1)\n            grid = F.affine_grid(theta, mu.size(), align_corners=False)\n            rotated = F.grid_sample(mu, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n            proj = rotated.sum(dim=-2) * dx * self.detector_scale\n            projections.append(proj)\n        sino = torch.stack(projections, dim=-2)\n        return self._resize_detector_axis(sino, self.num_detectors)\n\n    def _backproject_mu(self, sino, out_hw=None):\n        b, c, _, _ = sino.shape\n        h = self.img_dim if out_hw is None else int(out_hw[0])\n        w = self.img_dim if out_hw is None else int(out_hw[1])\n        dx = 2.0 / max(h, 1)\n        sino = self._resize_detector_axis(sino, w)\n        angles = self.angles.to(device=sino.device, dtype=sino.dtype)\n        back = torch.zeros((b, c, h, w), device=sino.device, dtype=sino.dtype)\n        for idx, angle in enumerate(angles):\n            proj = sino[:, :, idx, :]\n            slab = proj.unsqueeze(-2).expand(b, c, h, w) * dx * self.detector_scale\n            cos_a = torch.cos(angle)\n            sin_a = torch.sin(angle)\n            theta = torch.stack(\n                [\n                    torch.stack([cos_a, sin_a, torch.zeros_like(cos_a)]),\n                    torch.stack([-sin_a, cos_a, torch.zeros_like(cos_a)]),\n                ]\n            ).unsqueeze(0).expand(b, -1, -1)\n            grid = F.affine_grid(theta, slab.size(), align_corners=False)\n            back = back + F.grid_sample(slab, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n        return back\n\n    def line_integrals(self, image):\n        return self._forward_mu(self._mu(image))\n\n    def incident_counts(self, reference):\n        return torch.full_like(reference, fill_value=float(self.I0))\n\n    def H(self, image):\n        z = self.line_integrals(image)\n        return self.incident_counts(z) * torch.exp(-z).clamp_min(0.0)\n\n    def measure(self, image):\n        rate = self.H(image).clamp_min(0.0).clamp_max(1.0e12)\n        return torch.poisson(rate)\n\n    def _counts_to_line_integrals(self, counts):\n        i0 = self.incident_counts(counts)\n        frac = (counts.clamp_min(1.0) / i0.clamp_min(1.0)).clamp(min=1.0e-12, max=1.0)\n        return -torch.log(frac)\n\n    def _normalize_visual(self, image):\n        flat = image.flatten(1)\n        lo = flat.min(dim=1).values.reshape(-1, 1, 1, 1)\n        hi = flat.max(dim=1).values.reshape(-1, 1, 1, 1)\n        image01 = (image - lo) / (hi - lo).clamp_min(1.0e-8)\n        return image01 * 2.0 - 1.0\n\n    def measurement_visual(self, counts):\n        # The repo\'s "deg" image is a visual diagnostic, not the tensor optimized\n        # against. Show the log-transformed CT sinogram so it is interpretable.\n        z = self._counts_to_line_integrals(counts)\n        z = self._normalize_visual(z)\n        z = F.interpolate(z, size=(self.img_dim, self.img_dim), mode="bilinear", align_corners=False)\n        return self._repeat_channels(z)\n\n    def measurement_loss_per_sample(self, image, counts):\n        z = self.line_integrals(image)\n        i0 = self.incident_counts(z)\n        per_bin = i0 * torch.exp(-z).clamp_min(0.0) + counts * z\n        flat = per_bin.flatten(1)\n        if self.loss_reduction == "sum":\n            return flat.sum(dim=1)\n        return flat.mean(dim=1)\n\n    def measurement_loss(self, image, counts):\n        return self.measurement_loss_per_sample(image, counts).mean()\n\n    def measurement_loss_and_norm(self, image, counts):\n        per_sample = self.measurement_loss_per_sample(image, counts)\n        return per_sample.sum(), per_sample.clamp_min(1.0e-12).sqrt()\n\n    def H_pinv(self, counts):\n        z = self._counts_to_line_integrals(counts)\n        mu = self._backproject_mu(z) / max(self.num_angles, 1)\n        denom = max(self.attenuation_max - self.attenuation_min, 1.0e-12)\n        x01 = (mu - self.attenuation_min) / denom\n        x = (x01 * 2.0 - 1.0).clamp(-1.0, 1.0)\n        return self._repeat_channels(x)\n'

#@title Patch RED-diff Repo With Native-Count CT
import os
from pathlib import Path

os.chdir(REPO_DIR)

def replace_once(path: Path, old: str, new: str) -> None:
    text = path.read_text(encoding="utf-8")
    if old not in text:
        print(f"Pattern not found in {path}; leaving unchanged.")
        return
    path.write_text(text.replace(old, new, 1), encoding="utf-8")
    print(f"Patched {path}")

deg_path = Path(REPO_DIR) / "utils" / "degredations.py"
text = deg_path.read_text(encoding="utf-8")
if "class TransmissionCTNativeCountsH" not in text:
    marker = "\ndef build_degredation_model(cfg: DictConfig):\n"
    text = text.replace(marker, CT_DEGRADATION_CODE + marker, 1)
if 'elif deg in {"transmission_ct_native", "ct_native_counts"}:' not in text:
    branch = """    elif deg in {"transmission_ct_native", "ct_native_counts"}:
        H = TransmissionCTNativeCountsH(
            channels=c,
            img_dim=w,
            num_angles=int(_cfg_or_default(operator_cfg, "num_angles", 80)),
            num_detectors=int(_cfg_or_default(operator_cfg, "num_detectors", w)),
            I0=float(_cfg_or_default(operator_cfg, "I0", 10000.0)),
            attenuation_min=float(_cfg_or_default(operator_cfg, "attenuation_min", 0.0)),
            attenuation_max=float(_cfg_or_default(operator_cfg, "attenuation_max", 1.0)),
            detector_scale=float(_cfg_or_default(operator_cfg, "detector_scale", 1.0)),
            angle_offset_deg=float(_cfg_or_default(operator_cfg, "angle_offset_deg", 0.0)),
            loss_reduction=str(_cfg_or_default(operator_cfg, "loss_reduction", "mean")),
            device=device,
        )
"""
    text = text.replace('    if deg == "deno":\n        H = Denoising(c, w, device)\n', '    if deg == "deno":\n        H = Denoising(c, w, device)\n' + branch, 1)
deg_path.write_text(text, encoding="utf-8")
print(f"Native CT degradation ready in {deg_path}")

dm4ct_model_path = Path(REPO_DIR) / "models" / "dm4ct_diffusers.py"
dm4ct_model_path.write_text(r"""
import torch
import torch.nn as nn


class DM4CTDiffusersUNet(nn.Module):
    # Diffusers DDPM UNet adapter for the RED-diff repo model API.

    def __init__(self, model_id, local_files_only=False, torch_dtype="float32"):
        super().__init__()
        try:
            from diffusers import DDPMPipeline, DDPMScheduler, UNet2DModel
        except ImportError as exc:
            raise ImportError("Install diffusers to use the DM4CT checkpoint.") from exc

        dtype = getattr(torch, str(torch_dtype)) if torch_dtype else None
        load_kwargs = {"local_files_only": bool(local_files_only)}
        if dtype is not None:
            load_kwargs["torch_dtype"] = dtype

        errors = []
        try:
            pipe = DDPMPipeline.from_pretrained(model_id, **load_kwargs)
            self.unet = pipe.unet
            self.scheduler = pipe.scheduler
        except Exception as exc:
            errors.append(f"DDPMPipeline: {exc}")
            try:
                self.unet = UNet2DModel.from_pretrained(model_id, subfolder="unet", **load_kwargs)
                self.scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
            except Exception as sub_exc:
                errors.append(f"unet/scheduler subfolders: {sub_exc}")
                try:
                    self.unet = UNet2DModel.from_pretrained(model_id, **load_kwargs)
                    self.scheduler = DDPMScheduler(num_train_timesteps=1000)
                except Exception as bare_exc:
                    errors.append(f"bare UNet2DModel: {bare_exc}")
                    raise RuntimeError("Could not load DM4CT diffusers checkpoint:\n" + "\n".join(errors)) from bare_exc

        config = getattr(self.unet, "config", None)
        self.model_in_channels = int(getattr(config, "in_channels", 1))
        self.model_out_channels = int(getattr(config, "out_channels", self.model_in_channels))

    @staticmethod
    def _match_channels(x, channels):
        if x.shape[1] == channels:
            return x
        if channels == 1:
            return x.mean(dim=1, keepdim=True)
        if x.shape[1] > channels:
            return x[:, :channels]
        if x.shape[1] == 1:
            return x.repeat(1, channels, 1, 1)
        if channels % x.shape[1] == 0:
            return x.repeat(1, channels // x.shape[1], 1, 1)
        return x.mean(dim=1, keepdim=True).repeat(1, channels, 1, 1)

    def forward(self, x, timesteps, y=None):
        x_model = self._match_channels(x, self.model_in_channels)
        out = self.unet(x_model, timesteps).sample
        return self._match_channels(out, x.shape[1])
""", encoding="utf-8")
print(f"Wrote DM4CT diffusers model adapter to {dm4ct_model_path}")

model_init_path = Path(REPO_DIR) / "models" / "__init__.py"
text = model_init_path.read_text(encoding="utf-8")
old_model_load = """    model_ckpt = ckpt_path_adm(cfg.model.ckpt, cfg)
    logger.info(f"Loading model from {model_ckpt}..")
    model.load_state_dict(torch.load(model_ckpt, map_location=map_location), strict=False)
    print('asdas')
"""
new_model_load = """    if getattr(cfg.model, "ckpt", None):
        model_ckpt = ckpt_path_adm(cfg.model.ckpt, cfg)
        logger.info(f"Loading model from {model_ckpt}..")
        model.load_state_dict(torch.load(model_ckpt, map_location=map_location), strict=False)
    else:
        logger.info("Using model object without ADM checkpoint loading.")
"""
if old_model_load in text:
    text = text.replace(old_model_load, new_model_load, 1)
model_init_path.write_text(text, encoding="utf-8")
print("Model builder can use DM4CT diffusers checkpoints without ADM ckpt loading.")

configs_root = Path(REPO_DIR) / "_configs"
(configs_root / "model").mkdir(parents=True, exist_ok=True)
(configs_root / "dataset").mkdir(parents=True, exist_ok=True)

dm4ct_model_ref_yaml = str(DM4CT_MODEL_REF).replace("'", "''")
(configs_root / "model" / "dm4ct_lodochallenge_pixel.yaml").write_text(f"""# Runtime Colab config for the DM4CT LODO pixel-space CT prior.
_target_: models.dm4ct_diffusers.DM4CTDiffusersUNet
model_id: '{dm4ct_model_ref_yaml}'
local_files_only: false
torch_dtype: float32
""", encoding="utf-8")

(configs_root / "dataset" / "drive_ct_512.yaml").write_text("""# Colab-prepared CT float TIFFs.
name: "CT_Drive_512x512"
root: "/content/reddiff_ct_native_preprocessed"
split: "custom"
image_size: 512
channels: 3
meta_root: ""
transform: "diffusion"
subset_txt: ""
""", encoding="utf-8")

(configs_root / "dm4ct_ct512_uncond.yaml").write_text("""# RED-diff/DPS with the DM4CT CT prior and 512x512 CT slices.
defaults:
- dataset: drive_ct_512
- loader: imagenet256_ddrm
- model: dm4ct_lodochallenge_pixel
- classifier: none
- dist: localhost
- exp: default
- diffusion: linear1000
- algo: ddim
- _self_
""", encoding="utf-8")
print("Wrote DM4CT CT 512 configs.")

dataset_path = Path(REPO_DIR) / "datasets" / "imagenet.py"
text = dataset_path.read_text(encoding="utf-8")
if "def _load_ct_float_tiff_tensor" not in text:
    marker = "\n\nclass FlatImageDataset(Dataset):\n"
    helper = r"""

def _load_ct_float_tiff_tensor(path: str) -> torch.Tensor:
    arr = np.asarray(Image.open(path), dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[..., 0]
    if arr.size == 0:
        raise ValueError(f"Empty CT TIFF: {path}")
    finite = np.isfinite(arr)
    if not finite.all():
        arr = np.where(finite, arr, 0.0)
    if float(np.nanmin(arr)) < -0.01:
        arr = (arr + 1.0) * 0.5
    elif float(np.nanmax(arr)) > 1.01:
        max_val = 65535.0 if float(np.nanmax(arr)) > 255.0 else 255.0
        arr = arr / max_val
    arr = np.clip(arr, 0.0, 1.0).astype(np.float32)
    tensor = torch.from_numpy(arr).unsqueeze(0)
    return tensor.repeat(3, 1, 1)
"""
    text = text.replace(marker, helper + marker, 1)
old_getitem = """    def __getitem__(self, index: int):
        path, target = self.samples[index]
        sample = self.loader(path)
        if self.transform is not None:
            sample = self.transform(sample)

        name = os.path.splitext(os.path.basename(path))[0]
        return sample, target, {"class_id": self.class_id, "name": name, "index": index}
"""
new_getitem = """    def __getitem__(self, index: int):
        path, target = self.samples[index]
        if path.lower().endswith((\".tif\", \".tiff\")):
            sample = _load_ct_float_tiff_tensor(path)
        else:
            sample = self.loader(path)
            if self.transform is not None:
                sample = self.transform(sample)

        name = os.path.splitext(os.path.basename(path))[0]
        return sample, target, {\"class_id\": self.class_id, \"name\": name, \"index\": index}
"""
if old_getitem in text:
    text = text.replace(old_getitem, new_getitem, 1)
dataset_path.write_text(text, encoding="utf-8")
print("Flat custom dataset can read prepared CT float TIFFs without PNG quantization.")

dps_path = Path(REPO_DIR) / "algos" / "dps.py"
replace_once(
    dps_path,
    "            mat_norm = ((y_0 - H.H(x0_pred)).reshape(n, -1) ** 2).sum(dim=1).sqrt().detach()\n            mat = ((y_0 - H.H(x0_pred)).reshape(n, -1) ** 2).sum()\n\n            grad_term = torch.autograd.grad(mat, xt, retain_graph=True)[0]\n",
    "            if hasattr(H, \"measurement_loss_and_norm\"):\n                mat, mat_norm = H.measurement_loss_and_norm(x0_pred, y_0)\n                mat_norm = mat_norm.detach()\n            else:\n                mat_norm = ((y_0 - H.H(x0_pred)).reshape(n, -1) ** 2).sum(dim=1).sqrt().detach()\n                mat = ((y_0 - H.H(x0_pred)).reshape(n, -1) ** 2).sum()\n\n            grad_term = torch.autograd.grad(mat, xt, retain_graph=True)[0]\n",
)

reddiff_path = Path(REPO_DIR) / "algos" / "reddiff.py"
replace_once(
    reddiff_path,
    "            e_obs = y_0 - H.H(x0_pred)\n            loss_obs = (e_obs**2).mean()/2\n",
    "            if hasattr(H, \"measurement_loss\"):\n                loss_obs = H.measurement_loss(x0_pred, y_0)\n            else:\n                e_obs = y_0 - H.H(x0_pred)\n                loss_obs = (e_obs**2).mean()/2\n",
)

replace_once(
    deg_path,
    "def get_degreadation_image(y_0, H, cfg):\n    c, w = cfg.dataset.channels, cfg.dataset.image_size\n    pinv_y = H.H_pinv(y_0).reshape(-1, c, w, w)\n    return pinv_y\n",
    "def get_degreadation_image(y_0, H, cfg):\n    c, w = cfg.dataset.channels, cfg.dataset.image_size\n    if hasattr(H, \"measurement_visual\"):\n        return H.measurement_visual(y_0).reshape(-1, c, w, w)\n    pinv_y = H.H_pinv(y_0).reshape(-1, c, w, w)\n    return pinv_y\n",
)


In [ ]:
#@title Prepare CT Slices For The RED-diff Dataset Loader
import json
import os
import shutil
from pathlib import Path

import numpy as np
from PIL import Image

os.chdir(REPO_DIR)

valid_exts = {ext.strip().lower() for ext in VALID_EXTENSIONS.split(",") if ext.strip()}

def _load_ct_array(path: Path) -> np.ndarray:
    suffix = path.suffix.lower()
    if suffix in {".dcm", ".ima"}:
        import pydicom

        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        slope = float(getattr(ds, "RescaleSlope", 1.0))
        intercept = float(getattr(ds, "RescaleIntercept", 0.0))
        return arr * slope + intercept

    arr = np.asarray(Image.open(path), dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr

if DRIVE_CT_DATA_DIR.strip():
    input_root = Path(DRIVE_CT_DATA_DIR)
else:
    fallback_candidates = [
        Path(REPO_DIR) / "demo-samples" / "ct_l067_original_ima_subset",
        Path(REPO_DIR) / "demo_samples" / "ct_l067_original_ima_subset",
    ]
    input_root = next((path for path in fallback_candidates if path.exists()), None)
    if input_root is None:
        import urllib.request

        support_root = Path("/content/dm4ct_support_subset") / "ct_l067_original_ima_subset"
        support_root.mkdir(parents=True, exist_ok=True)
        base_url = (
            "https://raw.githubusercontent.com/Seif-Hussein/daps_test/"
            "codex-reddiff-colab-operators/demo-samples/ct_l067_original_ima_subset"
        )
        subset_files = [
            "L067_FD_1_SHARP_1.CT.0002.0009.2016.01.21.18.11.40.977560.404629207.IMA",
            "L067_FD_1_SHARP_1.CT.0002.0080.2016.01.21.18.11.40.977560.404630911.IMA",
            "L067_FD_1_SHARP_1.CT.0002.0528.2016.01.21.18.11.40.977560.404644046.IMA",
        ]
        for name in subset_files:
            dst = support_root / name
            if not dst.exists():
                urllib.request.urlretrieve(f"{base_url}/{name}", dst.as_posix())
        input_root = support_root

if not input_root.exists():
    raise FileNotFoundError(f"CT data folder not found: {input_root}")

all_files = sorted([p for p in input_root.rglob("*") if p.is_file() and p.suffix.lower() in valid_exts])
if not all_files:
    raise FileNotFoundError(f"No CT files with extensions {sorted(valid_exts)} under {input_root}")

start = int(DATA_START_IDX)
end = min(start + int(TOTAL_IMAGES), len(all_files))
selected_files = all_files[start:end]
if not selected_files:
    raise ValueError("Selected CT slice range is empty.")

if CT_VALUE_MIN.strip() and CT_VALUE_MAX.strip():
    value_min = float(CT_VALUE_MIN)
    value_max = float(CT_VALUE_MAX)
elif (not DRIVE_CT_DATA_DIR.strip()) and input_root.name == "ct_l067_original_ima_subset":
    value_min = -1024.0
    value_max = 3071.0
else:
    value_min = min(float(np.nanmin(_load_ct_array(path))) for path in all_files)
    value_max = max(float(np.nanmax(_load_ct_array(path))) for path in all_files)

if value_max <= value_min:
    raise ValueError(f"Invalid CT value range: {value_min}, {value_max}")

prep_root = Path(PREP_OUTPUT_DIR)
if prep_root.exists():
    shutil.rmtree(prep_root)
prep_root.mkdir(parents=True, exist_ok=True)

prepared = []
for idx, src in enumerate(selected_files):
    arr = _load_ct_array(src)
    arr01 = ((arr - value_min) / (value_max - value_min)).clip(0.0, 1.0)
    pil = Image.fromarray(arr01.astype(np.float32), mode="F").resize((512, 512), Image.BICUBIC)
    out = prep_root / f"ct_{idx:05d}_{src.stem}.tif"
    pil.save(out)
    prepared.append(out)

manifest = {
    "input_root": input_root.as_posix(),
    "selected_files": [path.as_posix() for path in selected_files],
    "prepared_files": [path.as_posix() for path in prepared],
    "ct_value_min": float(value_min),
    "ct_value_max": float(value_max),
    "prepared_format": "float32_tiff_0_1",
}
(prep_root / "source_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Prepared CT slices:")
for path in prepared:
    print(" -", path)
print(f"CT value range: [{value_min}, {value_max}]")


In [ ]:
#@title Build Single-Run Command
import json
import os
import shlex
import time
from pathlib import Path

os.chdir(REPO_DIR)

def strip_wrapped_quotes(text: str) -> str:
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {'"', "'"}:
        return text[1:-1].strip()
    return text

def parse_list(text: str):
    items = []
    for raw_item in text.replace("\n", ";").split(";"):
        raw_item = strip_wrapped_quotes(raw_item.strip())
        if raw_item:
            items.append(raw_item)
    return items

session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
run_slug = f"{ALGO_NAME}_transmission_ct_native_{session_tag}"
run_name = f"{RUN_NAME}_{session_tag}"

repo_dir = Path(REPO_DIR)
run_aux_root = repo_dir / "colab_runs"
subset_root = run_aux_root / "subsets"
subset_root.mkdir(parents=True, exist_ok=True)
subset_path = subset_root / f"{run_slug}.txt"

prepared_root = Path(PREP_OUTPUT_DIR)
prepared_files = sorted(prepared_root.glob("*.tif"))
subset_path.write_text("".join(f"{path.name} 0\n" for path in prepared_files), encoding="utf-8")

exp_root = run_aux_root
samples_root = exp_root / "samples" / run_slug
progress_path = samples_root / "progress.json"
metric_history_path = samples_root / "metric_history.json"
latest_log_path = run_aux_root / f"{run_slug}.log"
latest_pid_path = run_aux_root / f"{run_slug}.pid"

effective_batch_size = min(int(BATCH_SIZE), max(1, len(prepared_files)))

run_cmd = [
    PYTHON_BIN,
    "main.py",
    "-cn",
    BASE_CONFIG_NAME,
    "dataset=drive_ct_512",
    f"dataset.root={prepared_root.as_posix()}",
    f"dataset.subset_txt={subset_path.as_posix()}",
    f"algo={ALGO_NAME}",
    "algo.deg=transmission_ct_native",
    "classifier=none",
    "dist.num_processes_per_node=1",
    f"exp.root={exp_root.as_posix()}",
    f"exp.name={run_slug}",
    "exp.samples_root=samples",
    "exp.overwrite=true",
    f"exp.seed={SEED}",
    f"exp.num_steps={NUM_STEPS}",
    "exp.write_progress_json=true",
    "exp.progress_json=progress.json",
    f"+exp.progress_json_every={max(1, int(PROGRESS_JSON_EVERY))}",
    "exp.write_metrics_history_json=true",
    "exp.metrics_history_json=metric_history.json",
    f"+exp.sampling_metric_every={max(1, int(SAMPLING_METRIC_EVERY))}",
    f"exp.save_ori={'true' if SAVE_ORI else 'false'}",
    f"exp.save_deg={'true' if SAVE_DEG else 'false'}",
    f"exp.save_evolution={'true' if SAVE_EVOLUTION else 'false'}",
    f"exp.smoke_test={SMOKE_TEST}",
    f"loader.batch_size={effective_batch_size}",
    f"loader.num_workers={NUM_WORKERS}",
    "loader.shuffle=false",
    "loader.drop_last=false",
    "loader.pin_memory=true",
    "algo.sigma_y=0.0",
    "algo.operator.sigma=0.0",
    f"+algo.operator.I0={float(I0)}",
    f"+algo.operator.num_angles={int(NUM_ANGLES)}",
    f"+algo.operator.num_detectors={int(NUM_DETECTORS)}",
    f"+algo.operator.attenuation_min={float(ATTENUATION_MIN)}",
    f"+algo.operator.attenuation_max={float(ATTENUATION_MAX)}",
    f"+algo.operator.loss_reduction={CT_LOSS_REDUCTION}",
]

if ALGO_NAME == "dps":
    run_cmd.extend([
        "algo.awd=true",
        f"algo.grad_term_weight={float(DPS_GRAD_TERM_WEIGHT)}",
        f"algo.eta={float(DPS_DDIM_ETA)}",
    ])
elif ALGO_NAME == "reddiff":
    run_cmd.extend([
        "algo.awd=true",
        f"algo.grad_term_weight={float(REDDIFF_GRAD_TERM_WEIGHT)}",
        f"algo.obs_weight={float(REDDIFF_OBS_WEIGHT)}",
        f"algo.lr={float(REDDIFF_LR)}",
        f"algo.eta={float(REDDIFF_DDIM_ETA)}",
        f"algo.denoise_term_weight={REDDIFF_DENOISE_TERM_WEIGHT}",
        f"algo.sigma_x0={float(REDDIFF_SIGMA_X0)}",
    ])
else:
    raise ValueError(f"Unsupported ALGO_NAME: {ALGO_NAME}")

run_cmd.extend(parse_list(EXTRA_HYDRA_OVERRIDES))

print(f"Run slug: {run_slug}")
print(f"Algorithm: {ALGO_NAME}")
print(f"Prepared images: {len(prepared_files)}")
print(f"Effective batch size: {effective_batch_size}")
if ALGO_NAME == "dps":
    print(f"DPS guidance weight: {DPS_GRAD_TERM_WEIGHT}")
    print(f"DPS DDIM eta: {DPS_DDIM_ETA}")
else:
    print(f"REDDIFF grad weight: {REDDIFF_GRAD_TERM_WEIGHT}")
    print(f"REDDIFF obs weight: {REDDIFF_OBS_WEIGHT}")
print(f"Samples root: {samples_root}")
print(f"Progress JSON: {progress_path}")
print(f"Metric history JSON: {metric_history_path}")
print("\nCommand:")
print(" ".join(shlex.quote(part) for part in run_cmd))

run_context = {
    "run_slug": run_slug,
    "samples_root": samples_root.as_posix(),
    "progress_path": progress_path.as_posix(),
    "metric_history_path": metric_history_path.as_posix(),
    "metrics_history_path": metric_history_path.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "run_cmd": run_cmd,
}


In [ ]:
#@title Run
import subprocess
from pathlib import Path

latest_log_path = Path(run_context["latest_log_path"])
latest_pid_path = Path(run_context["latest_pid_path"])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

with latest_log_path.open("w", encoding="utf-8") as log_file:
    proc = subprocess.Popen(run_context["run_cmd"], stdout=log_file, stderr=subprocess.STDOUT)
    latest_pid_path.write_text(str(proc.pid), encoding="utf-8")
    print(f"Started PID {proc.pid}")
    print(f"Log: {latest_log_path}")
    ret = proc.wait()

print(f"Process exited with code {ret}")
if ret != 0:
    raise RuntimeError(f"Run failed with exit code {ret}. Check {latest_log_path}")


In [ ]:
#@title Show Artifacts
import json
from pathlib import Path

samples_root = Path(run_context["samples_root"])
progress_path = Path(run_context["progress_path"])
history_path = Path(run_context["metric_history_path"])
log_path = Path(run_context["latest_log_path"])

print(f"samples_root: {samples_root}")
print(f"progress.json: {progress_path.exists()}")
print(f"metric_history.json: {history_path.exists()}")

def _metric_history_rows(history):
    keys = ["step", "elapsed_seconds_per_image", "psnr", "ssim", "lpips"]
    if all(isinstance(history.get(key), list) for key in keys):
        count = len(history["step"])
        return [
            {key: history[key][idx] if idx < len(history[key]) else None for key in keys}
            for idx in range(count)
        ]
    return history.get("entries", [])

def _subsample_rows(rows, limit=25):
    if len(rows) <= limit:
        return rows
    if limit <= 1:
        return rows[-1:]
    indices = sorted({round(i * (len(rows) - 1) / (limit - 1)) for i in range(limit)})
    return [rows[idx] for idx in indices]

if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    rows = _metric_history_rows(history)
    print("\nmetric_history.json:")
    print(json.dumps({
        "status": history.get("status"),
        "full_history_rows": len(rows),
        "summary": history.get("summary"),
    }, indent=2))
    if rows:
        print("\nDisplayed rows are subsampled from the full metric_history.json:")
        print(json.dumps(_subsample_rows(rows), indent=2))
    else:
        print(json.dumps(history, indent=2)[:4000])

pngs = sorted(samples_root.rglob("*.png"))
print(f"png_count: {len(pngs)}")
for path in pngs[:20]:
    print(path)

if log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\nLog tail:")
    print("\n".join(lines[-int(LOG_TAIL_LINES):]))
